# Fatigue modeling

Ordinal and multiclass models with participant-level held-out test, GroupKFold CV, and Optuna tuning. Core logic lives in `src/modeling/`.

Tuning and CV use **train/val participants only**; held-out test participants never appear in Optuna or CV folds.


In [41]:
%pip install -q -r ../../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sys
from pathlib import Path

_src = Path('../../src').resolve()
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Ensure local src edits are picked up when re-running this cell.
for _mod in [k for k in list(sys.modules) if k == 'modeling' or k.startswith('modeling.')]:
    del sys.modules[_mod]

import pandas as pd
from modeling.baselines import (
    run_all_baseline_benchmarks,
    summarize_baseline_metrics,
)
from modeling.config import (
    DATA_PATH,
    HIGH_FATIGUE_THRESHOLD,
    N_CV_FOLDS,
    OPTUNA_TRIALS,
    STABILITY_SEEDS,
)
from modeling.data import load_fatigue_data, prepare_splits, split_summary_table
from modeling.registry import (
    HISTORY_ORDINAL_MODELS,
    ORDINAL_MODELS,
    RESIDUAL_ORDINAL_MODELS,
)
from modeling.runner import tune_and_benchmark_model
from modeling.validation import run_stability_study, summarize_stability


## 1. Load data and split

Participants are held out with a **stratified split** on per-participant high-fatigue rate (`prepare_splits(..., stratify=True)`) so train/val and test have similar class balance.

“We randomly assign whole participants to train/val or test, but we do it in a way that both groups contain a similar proportion of people who often report high fatigue — not just a random 8 people who might all happen to be high-fatigue reporters.”


In [52]:
df = load_fatigue_data('../../' + DATA_PATH)
bundle = prepare_splits(df)
y_high_fatigue = (df['fatigue_num'] >= HIGH_FATIGUE_THRESHOLD).astype(int)

print(f"Rows: {len(df):,}  Participants: {df['id'].nunique()}")
display(split_summary_table(bundle, y_high_fatigue))
print('Test participant ids:', sorted(bundle.test_ids))


Rows: 3,331  Participants: 42


,split,participants,rows,mean_fatigue,high_fatigue_rate
0,train_val,34,2646,2.419123,0.267196
1,test,8,685,2.816058,0.319708


Test participant ids: [np.int64(10), np.int64(18), np.int64(30), np.int64(37), np.int64(38), np.int64(42), np.int64(46), np.int64(50)]


Re-run the **init accumulators** cell below before a fresh partial run to clear prior tuned-model results.


In [ ]:
# Re-run this cell to clear accumulated model results before a fresh partial run.
ordinal_results = []
history_ordinal_results = []

ordinal_best_params = {}
history_best_params = {}
# multiclass_results = []
# multiclass_best_params = {}


## 2. Baseline benchmarks

Simple predictors evaluated with the same GroupKFold CV and held-out test protocol as the tuned models. Includes persistence baselines **`lag1_fatigue`** and **`expanding_mean`**.


In [ ]:
ordinal_baseline_results = run_all_baseline_benchmarks(bundle, task='ordinal', n_splits=N_CV_FOLDS)

ordinal_baseline_summary = summarize_baseline_metrics(ordinal_baseline_results, task='ordinal')

print('Ordinal baselines (test metrics)')
display(ordinal_baseline_summary[[c for c in ordinal_baseline_summary.columns if c.startswith('test_')]])


<!-- multiclass baselines -->


In [ ]:
pass


MAE: Mean Absolute Error;

RMSE: Root Mean Squared Error, measures the variation in residual/error

R2: how much variability is explained by the model

QWK: Quadratic Weighted Kappa. QWK measures the agreement between two raters—such as an AI and a human—on an ordered scale. It is designed to adjust for chance agreements and heavily penalize larger scoring discrepancies over minor ones.

## 3. Train/Tune models

Run only the model cells you need. Each cell tunes (Optuna) and benchmarks one model.

### Ordinal Regression

Continuous loss on `fatigue_num`, then round and clip to [0, 5].

#### `linear_regression`


In [55]:
_name = 'linear_regression'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordered_logistic  test_mae=1.2920


#### `ordinal_rf`


In [56]:
_name = 'ordinal_rf'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] ordinal_rf  test_mae=1.0818


#### `catboost_regressor`


In [57]:
_name = 'catboost_regressor'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


[ok] catboost_ordinal  test_mae=1.1693


### Ordinal Classification

Ordered likelihood or threshold structure on `fatigue_num` 0–5. Evaluated with the same MAE / QWK metrics as regression models.

#### `ordered_logistic`


In [ ]:
_name = 'ordered_logistic'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


#### `ordinal_forest`


In [ ]:
_name = 'ordinal_forest'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


#### `mixed_effects`


In [ ]:
_name = 'mixed_effects'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


#### `catboost_ordinal`


In [ ]:
_name = 'catboost_ordinal'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
ordinal_results.append(_result)
ordinal_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')


### History

History models share the same leakage-safe history features. **Ordinal** models predict `fatigue_num` (MAE). Multiclass sections below are placeholders.

Added **History features** (7 cols):
- fatigue lag1: Yesterday’s fatigue score
- fatigue EWMA: Exponentially weighted average of past fatigue; recent days count more
- fatigue expanding mean: Average fatigue on all earlier days for this person
- fatigue delta lag1: Change in fatigue, the worsening/improving trend
- activity_logsum_roll3_mean: Rolling mean of prior days' sum of log1p(lightly) + log1p(moderately) + log1p(very)
- calories_sum_roll3_mean: Recent typical daily calories burned
- very_roll3_mean: Recent typical “very active” minutes

#### `catboost_history`


In [62]:
_name = 'catboost_history'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    HISTORY_ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')
print(f'  params={_params}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


[ok] catboost_history  test_mae=0.8204
  params={'iterations': 422, 'depth': 4, 'learning_rate': 0.01680499858761428, 'l2_leaf_reg': 7.215428832529494, 'ewma_alpha': 0.22769026089922748, 'rolling_window': 3, 'loss_mode': 'rmse'}
  history_params={'ewma_alpha': 0.22769026089922748, 'rolling_window': 3}


**`catboost_history`** jointly tunes EWMA alpha, rolling window, CatBoost params, and loss mode (`rmse` vs `multiclass`).
- RMSE: regression with clip for out-of-boundary predictions
- Multiclass: six-class fatigue_num prediction (weighted F1)

#### `catboost_residual_expanding`


In [63]:
_name = 'catboost_residual_expanding'
_result, _params = tune_and_benchmark_model(
    _name,
    bundle,
    RESIDUAL_ORDINAL_MODELS,
    task='ordinal',
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
history_ordinal_results.append(_result)
history_best_params[_name] = _params
print(f'[ok] {_name}  test_mae={_result["test_metrics"]["mae"]:.4f}')
print(f'  params={_params}')
if _result.get('history_params'):
    print(f'  history_params={_result["history_params"]}')


[ok] catboost_residual_expanding  test_mae=0.8511
  params={'iterations': 142, 'depth': 6, 'learning_rate': 0.046552005417938476, 'l2_leaf_reg': 9.981399713597547, 'ewma_alpha': 0.46875245991163395, 'rolling_window': 5}
  history_params={'ewma_alpha': 0.46875245991163395, 'rolling_window': 5}


**`catboost_residual_expanding`**: `pred = clip(expanding_mean + CatBoost_residual)`. Jointly tunes EWMA alpha, rolling window, and CatBoost params (RMSE on residual only).

## 4. Results summary

Includes baselines plus only models whose §2 cells were executed. Ordinal CV summary shows **`cv_mae_std` only** (fold stability for MAE).

In [ ]:
def collect_summaries(results, task='ordinal'):
    cv_rows, test_rows = [], []
    if task == 'ordinal':
        metric_cols = ['mae', 'rmse', 'r2', 'qwk']
    else:
        metric_cols = ['weighted_f1', 'macro_f1', 'accuracy']
    for result in results:
        cv_mean = result['cv_summary'].loc['mean', metric_cols]
        cv_std = result['cv_summary'].loc['std', metric_cols]
        cv_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        test_row = {
            'model': result['name'],
            'best_params': str(result.get('best_params', {})),
        }
        for col in metric_cols:
            cv_row[f'cv_{col}'] = cv_mean[col]
            test_row[f'test_{col}'] = result['test_metrics'][col]
        if task == 'ordinal':
            cv_row['cv_mae_std'] = cv_std['mae']
        cv_rows.append(cv_row)
        test_rows.append(test_row)
    return pd.DataFrame(cv_rows).set_index('model'), pd.DataFrame(test_rows).set_index('model')

ordinal_results = globals().get('ordinal_results', [])
history_ordinal_results = globals().get('history_ordinal_results', [])
ordinal_best_params = globals().get('ordinal_best_params', {})
history_best_params = globals().get('history_best_params', {})

ran_ordinal = sorted(set(ordinal_best_params) | set(history_best_params))
print(f'Ran {len(ran_ordinal)} tuned ordinal models: {ran_ordinal}')

all_ordinal_results = ordinal_baseline_results + ordinal_results + history_ordinal_results
ordinal_cv_summary, ordinal_test_summary = collect_summaries(all_ordinal_results, task='ordinal')

print('Ordinal CV summary (baselines first)')
display(ordinal_cv_summary)
print('Ordinal held-out test summary')
display(ordinal_test_summary)


## 5. Model Stability

Run each cell separately. Each tuned model is re-fit with full Optuna on every seed. The summary cell adds `expanding_mean` (no tuning) and compares all three. Use `_stability_seeds = [42, 43, 44]` in the first cell for a quick smoke test.

Compare models by `test_mae_mean` directly (lower is better).


In [68]:
# Optional: use fewer seeds for a quick run
# _stability_seeds = [42, 43, 44]
_stability_seeds = STABILITY_SEEDS
stability_parts = globals().get('stability_parts', {})

_stability_history = run_stability_study(
    df,
    seeds=_stability_seeds,
    models=['catboost_history'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
stability_parts['catboost_history'] = _stability_history
print(f'[ok] catboost_history stability  rows={len(_stability_history)}')
display(_stability_history)


[ok] catboost_history stability  rows=10


,seed,model,test_mae,test_rmse,test_qwk,cv_mae_mean,cv_mae_std,best_params,n_test_participants
0,42,catboost_history,0.877372,1.195490,0.440979,0.884306,0.132824,"{'iterations': 406, 'depth': 9, 'learning_rate...",8
1,43,catboost_history,0.922287,1.200562,0.483819,0.845495,0.131629,"{'iterations': 415, 'depth': 6, 'learning_rate...",8
2,44,catboost_history,0.741085,1.031293,0.631220,0.884938,0.091741,"{'iterations': 364, 'depth': 4, 'learning_rate...",8
3,45,catboost_history,0.886503,1.190453,0.465993,0.884508,0.205782,"{'iterations': 483, 'depth': 10, 'learning_rat...",8
4,46,catboost_history,0.908946,1.202367,0.419293,0.869445,0.164425,"{'iterations': 150, 'depth': 4, 'learning_rate...",8
5,47,catboost_history,0.891107,1.235442,0.277650,0.853105,0.076717,"{'iterations': 337, 'depth': 4, 'learning_rate...",8
6,48,catboost_history,1.023769,1.285283,0.445742,0.830471,0.090976,"{'iterations': 420, 'depth': 4, 'learning_rate...",8
7,49,catboost_history,0.807249,1.133582,0.674197,0.866703,0.074576,"{'iterations': 459, 'depth': 4, 'learning_rate...",8
8,50,catboost_history,0.922953,1.211899,0.561874,0.846133,0.118751,"{'iterations': 398, 'depth': 5, 'learning_rate...",8
9,51,catboost_history,0.838996,1.103238,0.599255,0.865809,0.057732,"{'iterations': 311, 'depth': 4, 'learning_rate...",8


In [69]:
stability_parts = globals().get('stability_parts', {})

_stability_residual = run_stability_study(
    df,
    seeds=_stability_seeds,
    models=['catboost_residual_expanding'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)
stability_parts['catboost_residual_expanding'] = _stability_residual
print(f'[ok] catboost_residual_expanding stability  rows={len(_stability_residual)}')
display(_stability_residual)


[ok] catboost_residual_expanding stability  rows=10


,seed,model,test_mae,test_rmse,test_qwk,cv_mae_mean,cv_mae_std,best_params,n_test_participants
0,42,catboost_residual_expanding,0.830657,1.172059,0.477772,0.875398,0.087382,"{'iterations': 485, 'depth': 4, 'learning_rate...",8
1,43,catboost_residual_expanding,0.934018,1.258985,0.484608,0.860318,0.131247,"{'iterations': 306, 'depth': 8, 'learning_rate...",8
2,44,catboost_residual_expanding,0.728682,1.075448,0.670711,0.912506,0.115098,"{'iterations': 129, 'depth': 4, 'learning_rate...",8
3,45,catboost_residual_expanding,0.938650,1.207085,0.484381,0.866244,0.211305,"{'iterations': 361, 'depth': 4, 'learning_rate...",8
4,46,catboost_residual_expanding,1.004792,1.322574,0.292640,0.835963,0.105421,"{'iterations': 182, 'depth': 8, 'learning_rate...",8
5,47,catboost_residual_expanding,0.918330,1.262326,0.236914,0.853706,0.128942,"{'iterations': 131, 'depth': 4, 'learning_rate...",8
6,48,catboost_residual_expanding,1.047538,1.330713,0.431869,0.837017,0.126714,"{'iterations': 133, 'depth': 5, 'learning_rate...",8
7,49,catboost_residual_expanding,0.887974,1.177074,0.724315,0.885873,0.089777,"{'iterations': 388, 'depth': 6, 'learning_rate...",8
8,50,catboost_residual_expanding,0.966292,1.233560,0.597436,0.844083,0.142674,"{'iterations': 268, 'depth': 5, 'learning_rate...",8
9,51,catboost_residual_expanding,0.810931,1.175199,0.595154,0.880082,0.115257,"{'iterations': 499, 'depth': 8, 'learning_rate...",8


In [70]:
stability_parts = globals().get('stability_parts', {})
_seeds = globals().get('_stability_seeds', STABILITY_SEEDS)

_stability_baseline = run_stability_study(
    df,
    seeds=_seeds,
    models=['expanding_mean'],
    n_trials=OPTUNA_TRIALS,
    n_splits=N_CV_FOLDS,
)

parts = [_stability_baseline]
for _name in ['catboost_history', 'catboost_residual_expanding']:
    if _name in stability_parts:
        parts.append(stability_parts[_name])

stability_df = pd.concat(parts, ignore_index=True)
stability_summary = summarize_stability(stability_df)

print('Combined per-seed stability results')
display(stability_df)
print('Aggregated stability summary')
display(stability_summary)


Combined per-seed stability results


,seed,model,test_mae,test_rmse,test_qwk,cv_mae_mean,cv_mae_std,best_params,n_test_participants
0,42,expanding_mean,0.839416,1.167066,0.458234,0.917276,0.110738,{},8
1,43,expanding_mean,0.966276,1.262474,0.442917,0.882404,0.160850,{},8
2,44,expanding_mean,0.714729,1.042507,0.654058,0.944778,0.114536,{},8
3,45,expanding_mean,0.914110,1.200716,0.457333,0.896093,0.230233,{},8
4,46,expanding_mean,1.041534,1.354203,0.256255,0.870119,0.115845,{},8
5,47,expanding_mean,0.931034,1.270209,0.210295,0.893066,0.148471,{},8
6,48,expanding_mean,1.096774,1.362859,0.389816,0.856014,0.117818,{},8
7,49,expanding_mean,0.864909,1.153035,0.694839,0.909552,0.103643,{},8
8,50,expanding_mean,0.996790,1.262498,0.518076,0.876430,0.141788,{},8
9,51,expanding_mean,0.861152,1.200074,0.550023,0.911990,0.126854,{},8


Aggregated stability summary


,n_seeds,test_mae_mean,test_mae_std,test_mae_ci95_half,cv_mae_mean,cv_mae_std
model,,,,,,
catboost_history,10,0.882027,0.075677,0.046905,0.863091,0.114515
catboost_residual_expanding,10,0.906786,0.095398,0.059128,0.865119,0.125382
expanding_mean,10,0.922672,0.110187,0.068295,0.895772,0.137078
